# DG-01 — Micro-Hay Loss Landscape Observatory 12

Diagnostica **read-only** dei quattro checkpoint del factorial 11. Non addestra, non apre il test set e non rigenera il dataset. Misura gradienti per regime e blocco, SNR, conflitti, densità del target, superfici locali, interpolazioni e curvatura Hessiana.

Input Kaggle richiesti: `hay_micro_4c_event_enriched_v2.h5` e `hay_micro_orthogonal_factorial_11_complete.zip`.

In [ ]:
from pathlib import Path
import subprocess, sys

REPOSITORY_URL = 'https://github.com/Zagred47/LearningSingleCompartiment.git'

def valid_repository(path):
    path = Path(path)
    return ((path / 'pyproject.toml').is_file()
            and (path / 'src/hay_single_compartment').is_dir()
            and (path / 'notebooks/micro_loss_landscape_observatory_12.py').is_file())

candidates = [Path('/kaggle/working/LearningSingleCompartiment')]
candidates += [p.parent for p in Path('/kaggle/working').glob('**/pyproject.toml')]
candidates += [p.parent for p in Path('/kaggle/input').glob('**/pyproject.toml')]
REPO_ROOT = next((p.resolve() for p in candidates if valid_repository(p)), None)
if REPO_ROOT is None:
    base = Path('/kaggle/working/LearningSingleCompartiment_dg01')
    target, suffix = base, 1
    while target.exists():
        target = Path(f'{base}_{suffix}'); suffix += 1
    subprocess.check_call(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(target)])
    REPO_ROOT = target.resolve()
elif (REPO_ROOT / '.git').is_dir() and str(REPO_ROOT).startswith('/kaggle/working/'):
    subprocess.check_call(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'main'])
src = str(REPO_ROOT / 'src')
if src not in sys.path: sys.path.insert(0, src)
print('Repository:', REPO_ROOT)

In [ ]:
import os
from pathlib import Path

print('HDF5 disponibili:')
for path in sorted(Path('/kaggle/input').glob('**/*.h5')): print(' ', path)
print('ZIP disponibili:')
for path in sorted(Path('/kaggle/input').glob('**/*.zip')): print(' ', path)

# La discovery sceglie automaticamente gli input corretti. Se necessario:
# os.environ['HAY_DG01_DATASET'] = '/kaggle/input/.../hay_micro_4c_event_enriched_v2.h5'
# os.environ['HAY_DG01_FACTORIAL'] = '/kaggle/input/.../hay_micro_orthogonal_factorial_11_complete.zip'
os.environ['HAY_DG01_OUTPUT'] = '/kaggle/working/hay_micro_loss_landscape_12'

# Preflight tecnico rapido: non usarlo per le conclusioni scientifiche.
# os.environ['HAY_DG01_RUNS'] = 'gru_mse'
# os.environ['HAY_DG01_WINDOWS_PER_VIEW'] = '2'
# os.environ['HAY_DG01_LOCAL_GRID_POINTS'] = '3'
# os.environ['HAY_DG01_INTERPOLATION_POINTS'] = '3'
# os.environ['HAY_DG01_HESSIAN_WINDOWS'] = '1'
# os.environ['HAY_DG01_HESSIAN_ITERATIONS'] = '2'
# os.environ['HAY_DG01_HUTCHINSON_PROBES'] = '2'

# Solo per isolare eventuali problemi tecnici delle parti costose:
# os.environ['HAY_DG01_SKIP_LANDSCAPE'] = '1'
# os.environ['HAY_DG01_SKIP_HESSIAN'] = '1'

In [ ]:
import runpy
result = runpy.run_path(str(REPO_ROOT / 'notebooks/micro_loss_landscape_observatory_12.py'))
OUTPUT_DIR = Path(result['OUTPUT'])
ZIP_PATH = Path(result['ZIP_PATH'])
print('Output:', OUTPUT_DIR)
print('ZIP:', ZIP_PATH)

In [ ]:
import json, pandas as pd
from IPython.display import Image, display

display(pd.read_csv(OUTPUT_DIR / 'gradient_summary.csv'))
display(pd.read_csv(OUTPUT_DIR / 'gradient_cosines.csv').query("parameter_block == 'all'"))
if (OUTPUT_DIR / 'hessian_summary.csv').exists():
    display(pd.read_csv(OUTPUT_DIR / 'hessian_summary.csv'))
display(pd.Series(json.loads((OUTPUT_DIR / 'decision.json').read_text())))
for name in ['gradient_cosine_heatmaps.png', 'gradient_norm_and_snr.png', 'local_event_landscapes.png', 'interpolation_profiles.png', 'target_density_vs_error.png']:
    path = OUTPUT_DIR / 'figures' / name
    if path.exists():
        print(name); display(Image(filename=str(path)))

In [ ]:
from IPython.display import FileLink, Javascript, display
import base64
display(FileLink(str(ZIP_PATH)))
size_mib = ZIP_PATH.stat().st_size / 2**20
if size_mib <= 80:
    encoded = base64.b64encode(ZIP_PATH.read_bytes()).decode('ascii')
    filename = ZIP_PATH.name
    display(Javascript(f'''
    const binary = atob('{encoded}');
    const bytes = new Uint8Array(binary.length);
    for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
    const blob = new Blob([bytes], {{type: 'application/zip'}});
    const url = URL.createObjectURL(blob);
    const anchor = document.createElement('a');
    anchor.href = url; anchor.download = '{filename}';
    document.body.appendChild(anchor); anchor.click(); anchor.remove();
    setTimeout(() => URL.revokeObjectURL(url), 60000);
    '''))
    print('Download avviato:', ZIP_PATH, f'({size_mib:.1f} MiB)')
else:
    print(f'ZIP da {size_mib:.1f} MiB: usa il FileLink o il pannello Files di Kaggle.')